# COMPARISON — Temporal-First Graph Network (TFGN) ablation ladder

Implements `DOCS/temporal-first-ablation.md`'s 2026-08-24 "Evaluation & Comparison
Protocol" addendum over the ladder in `DOCS/flipped/PLAN.md` Phase 4 /
`CLASSIFIER/experiments/temporal_first.yaml`. Every section through Tier 3 reads only
`run_summary["oof"]` / `oof_predictions.csv` — **never** `test_*` / `ext_*` keys. The
one frozen in-domain + one frozen OASIS-3 read live in the final section only, gated
behind an explicit flag (`RUN_FROZEN_READ`), so re-running this notebook does not
accidentally spend the ladder's one test read before it is meant to.

`source_experiment`-style: no training here — every number comes from
`outputs/<rung>-seed{42..45}/latest/run_summary.json` (`adapters.explain.resolve_source_run`),
written by `LONGITUDINAL_COMMON_DELCODE.ipynb`'s runs.

In [1]:
# === Papermill parameters ===
EXPERIMENT_ID = None
MODE = None
MODEL = None
SEED = 42
WANDB_ENABLED = False
OUTPUT_DIR = None
RUN_DIR = None
RUN_NAME = None
# Tier-4 is gated: only flip this (and set FROZEN_WINNER_ID) once the FULL chain
# (S1 -> S1c_recon_random -> S2 -> S3 -> S4 -> S5 -> SENS) has reported against
# Tier 2/Tier 3 -- not merely once S1c-random has (DOCS/temporal-first-ablation.md
# Tier-4 gate, restated 2026-08-24).
RUN_FROZEN_READ = False
FROZEN_WINNER_ID = None       # the frozen S1-lineage winner's id prefix (no seed suffix),
                               # e.g. 'tfgn-s5-dualscore-pooled' once SENS has reported.
SECONDARY_SENSITIVITY_ID = None  # optional: a single id prefix, OR a list of id prefixes,
                                  # e.g. ['tfgn-s1b-ssl-pooled', 'tfgn-s5-dualscore-pooled'] --
                                  # secondary reads taken in this SAME pass and reported side
                                  # by side, never substituted for the primary. S5's read is
                                  # the interpretability layer's number (kept regardless of
                                  # AUC, PLAN.md section A), not a competing endpoint -- its
                                  # label makes that explicit below.


In [2]:
# Parameters
RUN_FROZEN_READ = True
FROZEN_WINNER_ID = "tfgn-s1-flip-pooled"
SECONDARY_SENSITIVITY_ID = ["tfgn-s1b-ssl-pooled", "tfgn-s5-dualscore-pooled"]


## Pipeline overview

Resolve each rung's 4 seed runs -> Tier 1 floors -> Tier 2 rung table + stopping rule (paired seed-level OOF ΔAUC) -> Tier 3 vetoes -> Tier 4 frozen reads (gated).

In [3]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))


In [4]:
# reproducibility seeding -- must run before datasets / models.
from SHARED.seeding import set_seed, make_rng, make_torch_generator
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)


In [5]:
import json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from adapters.explain import resolve_source_run
from common.comparison import paired_bootstrap_ci

warnings.filterwarnings('ignore')
print('Imports OK')


Imports OK


## Configuration — rung registry

One entry per ladder rung; extend this dict as S1c-SENS land (`DOCS/temporal-first-ablation.md` "The arms") -- nothing else in this notebook needs to change. `S0_demo` is the new Tier-1 demographics floor (`tfgn-s0-demo-pooled`); it has no runs yet until dispatched.

In [6]:
SEEDS = [42, 43, 44, 45]

# name -> registry id prefix (seed appended as '-seed{42,43,44,45}').
RUNG_PREFIXES = {
    'S0a_logreg_drift':   'tfgn-s0-logreg-drift-pooled',
    'S0_demo':            'tfgn-s0-demo-pooled',
    'S0b_gelstm_frozen':  'tfgn-s0-gelstm-frozen-pooled',
    'S0c_gelstm_random':  'tfgn-s0-gelstm-random-pooled',
    'S0d_braintokengt':   'tfgn-s0-braintokengt-pooled',
    'S1_flip':            'tfgn-s1-flip-pooled',
    'S1b_ssl':             'tfgn-s1b-ssl-pooled',
    'S1c_recon_invalid':   'tfgn-s1c-recon-pooled',           # SUPERSEDED 2026-08-24 -- built on the
                                                                # since-reversed pretrained_finetuned fork;
                                                                # recorded as undecidable, kept ONLY out of
                                                                # RUNG_CHAIN/HEADLINE_CONTRASTS below (see the
                                                                # markdown cell after this one).
    'S1c_recon_random':    'tfgn-s1c-recon-random-pooled',    # protocol-valid re-run, node_lstm_init=random
    # S2-SENS all branch from S1 (not from S1c_recon_random, which the stopping rule
    # rejected) -- their Tier-2 comparison is against S1_flip directly, computed in
    # the section-F/G-style ad-hoc cells in DOCS/flipped/PLAN.md, not via RUNG_CHAIN
    # (which models a sequential chain and would misattribute these deltas against
    # S1c_recon_random if simply appended). Left out of RUNG_CHAIN/HEADLINE_CONTRASTS
    # deliberately; their OOF rows still surface in RUNG_SUMMARY_TABLE below.
    'S2_gate':             'tfgn-s2-gate-pooled',
    'S3_fusion':           'tfgn-s3-fusion-pooled',           # VOID -- inert knob, bit-identical to S1 (PLAN.md section B)
    'S4_attnpool':         'tfgn-s4-attnpool-pooled',
    'S5_dualscore':        'tfgn-s5-dualscore-pooled',        # kept regardless of AUC -- interpretability layer (PLAN.md section A)
    'SENS':                'tfgn-sens-minvisits3-pooled',     # min_visits=3, N=140 -- not fold-matched vs S1 (PLAN.md section J)
}

# The pre-registered chain (DOCS/temporal-first-ablation.md "The arms"), CORRECTED
# 2026-08-24 per the "S1b fork decision" correcting addendum: S1b is dropped by
# Tier 2's one-SE tie-breaker (see the tie-breaker cell below) and is therefore NOT
# in the primary chain -- it is scored separately as a sensitivity arm. S1c_recon
# (the original, pretrained_finetuned run) is likewise excluded from the chain --
# it tests an unregistered configuration, not the S1c question. The chain's Tier-2
# stopping-rule comparison is each rung against the one immediately before it here.
RUNG_CHAIN = ['S0c_gelstm_random', 'S1_flip', 'S1c_recon_random']  # extend: ..., 'S2_gate', ...

# Sensitivity arms: reported (fold-matched AND pooled OOF Δ vs their reference rung)
# but never part of RUNG_CHAIN and never selected on. S1b's own OOF rows still
# appear in RUNG_SUMMARY_TABLE via RUNG_PREFIXES above.
SENSITIVITY_CONTRASTS = {
    'S1b_vs_S1': ('S1_flip', 'S1b_ssl'),  # the corrected fork read -- see the tie-breaker cell
}

HEADLINE_CONTRASTS = {
    'S0c_vs_S1': ('S0c_gelstm_random', 'S1_flip'),
    'S0b_vs_S1c': ('S0b_gelstm_frozen', 'S1c_recon_random'),  # the protocol-valid re-run, not S1c_recon_invalid
}


**S1c status note (2026-08-24).** `tfgn-s1c-recon-pooled` (`S1c_recon_invalid` above)
was built with `node_lstm_init: pretrained_finetuned`, inheriting the S1b fork decision
as it stood before the correcting addendum below reversed it. That run tests an
unregistered configuration and is recorded as **undecidable**, not as a result for or
against the flip -- see `DOCS/temporal-first-ablation.md` "S1c (2026-08-24 run) --
recorded as undecidable". It is intentionally excluded from `RUNG_CHAIN` and from
`HEADLINE_CONTRASTS`; only `S1c_recon_random` (`node_lstm_init: random`, the
protocol-valid re-run) participates in either. Its OOF rows remain visible in
`RUNG_SUMMARY_TABLE` for the record, labelled accordingly.

In [7]:
def resolve_rung_runs(prefix):
    """exp_id -> run_dir for every seed of one rung; missing runs are skipped, not fatal
    (this notebook must stay runnable before every rung/seed has been dispatched)."""
    run_dirs = {}
    for seed in SEEDS:
        exp_id = f'{prefix}-seed{seed}'
        try:
            run_dirs[exp_id] = resolve_source_run(exp_id, classifier_root=model_root)
        except FileNotFoundError:
            print(f'  [skip] {exp_id}: no run yet')
    return run_dirs


RUNG_RUN_DIRS = {name: resolve_rung_runs(prefix) for name, prefix in RUNG_PREFIXES.items()}
for name, dirs in RUNG_RUN_DIRS.items():
    print(f'{name:22s} {len(dirs)}/{len(SEEDS)} seeds resolved')


S0a_logreg_drift       4/4 seeds resolved
S0_demo                4/4 seeds resolved
S0b_gelstm_frozen      4/4 seeds resolved
S0c_gelstm_random      4/4 seeds resolved
S0d_braintokengt       4/4 seeds resolved
S1_flip                4/4 seeds resolved
S1b_ssl                4/4 seeds resolved
S1c_recon_invalid      4/4 seeds resolved
S1c_recon_random       4/4 seeds resolved
S2_gate                4/4 seeds resolved
S3_fusion              4/4 seeds resolved
S4_attnpool            4/4 seeds resolved
S5_dualscore           4/4 seeds resolved
SENS                   4/4 seeds resolved


In [8]:
def load_summary(run_dir):
    """None (not a raised error) if the run hasn't written run_summary.json yet --
    a run still 'running' on the other host is a normal state this notebook must
    tolerate, not treat as missing/broken."""
    path = run_dir / 'run_summary.json'
    return json.loads(path.read_text()) if path.is_file() else None


def load_calibration(run_dir):
    path = run_dir / 'calibration.json'
    return json.loads(path.read_text()) if path.is_file() else {}


def rung_oof_rows(run_dirs):
    """One row per seed with that seed's oof.* metrics (empty 'oof' -> pre-addendum,
    not-yet-re-run artifact -- flagged, not silently dropped)."""
    rows = []
    for exp_id, run_dir in run_dirs.items():
        summary = load_summary(run_dir)
        if summary is None:
            print(f'  [in-flight] {exp_id}: no run_summary.json yet (run still running?) -- skipped.')
            continue
        oof = summary.get('oof')
        if not oof:
            print(f'  [stale] {exp_id}: no run_summary["oof"] -- re-run under the '
                  '2026-08-24 addendum contract before trusting this rung\'s table row.')
            continue
        cal = load_calibration(run_dir)
        row = {'exp_id': exp_id, 'seed': int(exp_id.rsplit('seed', 1)[-1]),
               'cohort_probe_auc': summary.get('cohort_probe_auc'),
               'ece_oof_cal': cal.get('ece_oof_cal')}
        row.update(oof)
        rows.append(row)
    return pd.DataFrame(rows)


## Tier 1 — floor gates

In [9]:
# Demographics floor: tfgn-s0-demo-pooled (feature_set='demo', [age, sex] only).
DEMO_FLOOR = rung_oof_rows(RUNG_RUN_DIRS.get('S0_demo', {}))
if not DEMO_FLOOR.empty:
    print('Demographics floor (age+sex only), OOF AUC:',
          f"{DEMO_FLOOR['oof_auc'].mean():.4f} +/- {DEMO_FLOOR['oof_auc'].std():.4f}")
else:
    print('Demographics floor: no runs yet (dispatch tfgn-s0-demo-pooled-seed{42..45}).')

# SSL persistence baseline -- already computed by LONGITUDINAL_TFGN_SSL_POOLED.ipynb
# itself; nothing to compute here, just surface it.
_p2_dir = resolve_source_run('tfgn-nodelstm-ssl-pooled', classifier_root=model_root)
_p2_summary = load_summary(_p2_dir) or {}
PERSISTENCE_BASELINE = _p2_summary.get('persistence_baseline', {})
print('P2 SSL persistence baseline:', PERSISTENCE_BASELINE)


Demographics floor (age+sex only), OOF AUC: 0.5296 +/- 0.0000
P2 SSL persistence baseline: {'val_loss_mse': 0.063011, 'n_val_subjects': 103, 'lstm_best_val_loss': 0.045883, 'improvement_pct': 27.2, 'note': 'x(t+1)=x(t) baseline on the same val split; LSTM at init (epoch 0, val_loss=0.0658) started worse than persistence, training pulled it 27.2% below.'}


## Tier 2 — rung table (OOF only) + stopping rule

In [10]:
RUNG_TABLES = {name: rung_oof_rows(dirs) for name, dirs in RUNG_RUN_DIRS.items()}

_cols = ['oof_auc', 'oof_pr_auc', 'oof_balanced_accuracy', 'oof_static_n1_auc', 'cohort_probe_auc']
summary_rows = []
for name, df in RUNG_TABLES.items():
    if df.empty:
        summary_rows.append({'rung': name, 'n_seeds': 0})
        continue
    row = {'rung': name, 'n_seeds': len(df)}
    for c in _cols:
        if c in df.columns:
            row[f'{c}_mean'] = df[c].mean()
            row[f'{c}_sd'] = df[c].std()
    cohort_cols = [c for c in df.columns if c.startswith('oof_auc_') and c != 'oof_auc']
    for c in cohort_cols:
        row[f'{c}_mean'] = df[c].mean()
    summary_rows.append(row)

RUNG_SUMMARY_TABLE = pd.DataFrame(summary_rows).set_index('rung')
RUNG_SUMMARY_TABLE


,n_seeds,oof_auc_mean,oof_auc_sd,oof_pr_auc_mean,oof_pr_auc_sd,oof_balanced_accuracy_mean,oof_balanced_accuracy_sd,oof_static_n1_auc_mean,oof_static_n1_auc_sd,cohort_probe_auc_mean,cohort_probe_auc_sd,oof_auc_adni_mean,oof_auc_delcode_mean
rung,,,,,,,,,,,,,
S0a_logreg_drift,4,0.705321,0.000000,0.544264,0.000000,0.677620,0.000000,0.565260,0.000000,NaN,NaN,0.556359,0.912861
S0_demo,4,0.529574,0.000000,0.398237,0.000000,0.513921,0.000000,0.529574,0.000000,NaN,NaN,0.516184,0.518639
S0b_gelstm_frozen,4,0.718606,0.033361,0.621836,0.031719,0.650519,0.023144,0.469896,0.027054,NaN,NaN,0.497144,0.917754
S0c_gelstm_random,4,0.562513,0.029217,0.418254,0.023505,0.563149,0.023772,0.514001,0.021455,NaN,NaN,0.534772,0.606652
S0d_braintokengt,4,0.620734,0.033849,0.496457,0.041313,0.588907,0.054540,0.535263,0.056132,NaN,NaN,0.619383,0.621738
S1_flip,4,0.748816,0.003284,0.656420,0.014228,0.709270,0.017515,0.491873,0.006432,0.859975,0.007444,0.652609,0.874068
S1b_ssl,4,0.750194,0.012486,0.640666,0.026191,0.703661,0.012956,0.507226,0.010843,0.889796,0.027270,0.662414,0.870690
S1c_recon_invalid,4,0.550703,0.041615,0.400892,0.026212,0.530881,0.014018,0.538160,0.026092,0.860954,0.050158,0.556359,0.533551
S1c_recon_random,4,0.543292,0.031132,0.399918,0.032479,0.547656,0.010084,0.521341,0.023349,0.860028,0.026924,0.533535,0.558947


In [11]:
def per_fold_auc(run_dir):
    """fold -> OOF AUC, from oof_predictions.csv. The StratifiedGroupKFold split in
    common.crossval.run_kfold_cv takes no seed/shuffle, so fold i is the SAME subject
    group across every seed and every rung of one dataset -- this is what makes
    fold-matched pairing across arms/seeds valid."""
    path = run_dir / 'oof_predictions.csv'
    if not path.is_file():
        return {}
    df = pd.read_csv(path)
    out = {}
    for fold, sub in df.groupby('fold'):
        if sub['label'].nunique() > 1:
            out[int(fold)] = roc_auc_score(sub['label'], sub['prob'])
    return out


def stopping_rule(rung_k_dirs, rung_km1_dirs):
    """mean(Delta) / SE(Delta) of the seed-level mean paired fold-matched OOF ΔAUC
    (rung k vs rung k-1) -- DOCS/temporal-first-ablation.md 'The stopping rule',
    Tier 2's own definition. This is the statistic that governs keep/drop."""
    seed_means = []
    for exp_id_k, dir_k in rung_k_dirs.items():
        seed = exp_id_k.rsplit('seed', 1)[-1]
        matches = [d for eid, d in rung_km1_dirs.items() if eid.endswith(f'seed{seed}')]
        if not matches:
            continue
        auc_k, auc_km1 = per_fold_auc(dir_k), per_fold_auc(matches[0])
        common_folds = sorted(set(auc_k) & set(auc_km1))
        if not common_folds:
            continue
        seed_means.append(float(np.mean([auc_k[f] - auc_km1[f] for f in common_folds])))
    if len(seed_means) < 2:
        return {'mean': float('nan'), 'se': float('nan'), 'ratio': float('nan'),
                'n_seeds': len(seed_means), 'seed_means': seed_means, 'kept': None}
    mean = float(np.mean(seed_means))
    se = float(np.std(seed_means, ddof=1) / np.sqrt(len(seed_means)))
    return {'mean': mean, 'se': se, 'ratio': (mean / se) if se > 0 else float('nan'),
            'n_seeds': len(seed_means), 'seed_means': seed_means, 'kept': mean > se}


def pooled_stopping_rule(rung_k_dirs, rung_km1_dirs):
    """Secondary sanity statistic (2026-08-24 Tier-2 clarification,
    DOCS/temporal-first-ablation.md): each seed's pooled run_summary['oof']['oof_auc']
    differenced directly, WITHOUT fold pairing -- noisier than stopping_rule() above,
    never the keep/drop statistic on its own. Reported alongside it; the two must be
    printed together whenever they disagree in sign or keep/drop, per that
    clarification, rather than reporting only the number that supports a conclusion."""
    def pooled_auc(run_dir):
        p = run_dir / 'run_summary.json'
        if not p.is_file():
            return None
        s = json.loads(p.read_text())
        return s.get('oof', {}).get('oof_auc')

    seed_deltas = []
    for exp_id_k, dir_k in rung_k_dirs.items():
        seed = exp_id_k.rsplit('seed', 1)[-1]
        matches = [d for eid, d in rung_km1_dirs.items() if eid.endswith(f'seed{seed}')]
        if not matches:
            continue
        auc_k, auc_km1 = pooled_auc(dir_k), pooled_auc(matches[0])
        if auc_k is None or auc_km1 is None:
            continue
        seed_deltas.append(auc_k - auc_km1)
    if len(seed_deltas) < 2:
        return {'mean': float('nan'), 'se': float('nan'), 'ratio': float('nan'),
                'n_seeds': len(seed_deltas), 'seed_deltas': seed_deltas, 'kept': None}
    mean = float(np.mean(seed_deltas))
    se = float(np.std(seed_deltas, ddof=1) / np.sqrt(len(seed_deltas)))
    return {'mean': mean, 'se': se, 'ratio': (mean / se) if se > 0 else float('nan'),
            'n_seeds': len(seed_deltas), 'seed_deltas': seed_deltas, 'kept': mean > se}


def report_contrast(label, a, b):
    """Print both Tier-2 statistics for rung b vs rung a; flag disagreement (2026-08-24
    Tier-2 clarification -- never report only the statistic that supports a conclusion)."""
    fm = stopping_rule(RUNG_RUN_DIRS.get(b, {}), RUNG_RUN_DIRS.get(a, {}))
    pl = pooled_stopping_rule(RUNG_RUN_DIRS.get(b, {}), RUNG_RUN_DIRS.get(a, {}))
    fm_verdict = ('worth carrying forward' if fm['kept'] else
                  'undetectable at this sample size' if fm['kept'] is not None else
                  'not enough seeds resolved yet')
    print(f"  {label} ({b} vs {a})")
    print(f"    fold-matched: mean(D)={fm['mean']:.5f} SE(D)={fm['se']:.5f} "
          f"ratio={fm['ratio']:.2f} n_seeds={fm['n_seeds']} -> {fm_verdict}")
    print(f"    pooled:       mean(D)={pl['mean']:.5f} SE(D)={pl['se']:.5f} "
          f"ratio={pl['ratio']:.2f} n_seeds={pl['n_seeds']}")
    if fm['kept'] is not None and pl['kept'] is not None and fm['kept'] != pl['kept']:
        print(f"    *** DISAGREEMENT: fold-matched kept={fm['kept']} but pooled kept={pl['kept']} -- "
              f"apply the Tier-2 one-SE tie-breaker (next cell) rather than picking one reading. ***")
    return fm, pl


print('Chain-adjacent stopping-rule decisions (OOF):')
for k in range(1, len(RUNG_CHAIN)):
    a, b = RUNG_CHAIN[k - 1], RUNG_CHAIN[k]
    report_contrast(f'{b} vs {a}', a, b)

print()
print('Sensitivity contrasts (not part of RUNG_CHAIN, never selected on):')
for label, (a, b) in SENSITIVITY_CONTRASTS.items():
    report_contrast(label, a, b)

print()
print('Headline contrasts (isolate the flip itself):')
for label, (a, b) in HEADLINE_CONTRASTS.items():
    report_contrast(label, a, b)


Chain-adjacent stopping-rule decisions (OOF):


  S1_flip vs S0c_gelstm_random (S1_flip vs S0c_gelstm_random)
    fold-matched: mean(D)=0.10828 SE(D)=0.01490 ratio=7.27 n_seeds=4 -> worth carrying forward
    pooled:       mean(D)=0.18630 SE(D)=0.01450 ratio=12.85 n_seeds=4
  S1c_recon_random vs S1_flip (S1c_recon_random vs S1_flip)
    fold-matched: mean(D)=-0.15869 SE(D)=0.00993 ratio=-15.98 n_seeds=4 -> undetectable at this sample size
    pooled:       mean(D)=-0.20552 SE(D)=0.01498 ratio=-13.72 n_seeds=4

Sensitivity contrasts (not part of RUNG_CHAIN, never selected on):


  S1b_vs_S1 (S1b_ssl vs S1_flip)
    fold-matched: mean(D)=0.01199 SE(D)=0.00186 ratio=6.46 n_seeds=4 -> worth carrying forward
    pooled:       mean(D)=0.00138 SE(D)=0.00739 ratio=0.19 n_seeds=4
    *** DISAGREEMENT: fold-matched kept=True but pooled kept=False -- apply the Tier-2 one-SE tie-breaker (next cell) rather than picking one reading. ***

Headline contrasts (isolate the flip itself):


  S0c_vs_S1 (S1_flip vs S0c_gelstm_random)
    fold-matched: mean(D)=0.10828 SE(D)=0.01490 ratio=7.27 n_seeds=4 -> worth carrying forward
    pooled:       mean(D)=0.18630 SE(D)=0.01450 ratio=12.85 n_seeds=4
  S0b_vs_S1c (S1c_recon_random vs S0b_gelstm_frozen)
    fold-matched: mean(D)=-0.15296 SE(D)=0.01327 ratio=-11.53 n_seeds=4 -> undetectable at this sample size
    pooled:       mean(D)=-0.17531 SE(D)=0.02443 ratio=-7.18 n_seeds=4


## Tier 2 tie-breaker -- one-SE simplicity rule

Pre-registered in `DOCS/temporal-first-ablation.md` Tier 2, before any of the numbers
below existed: "among kept arms, prefer the simplest configuration within one SE of the
best." This cell computes the selection, it does not assert it -- the S1-over-S1b
decision (and any future tie) falls out of this rule mechanically, not by hand.

In [12]:
def one_se_tie_breaker(candidates):
    """candidates: {name: (pooled_oof_auc_mean, pooled_oof_auc_se, complexity_rank)}
    where a LOWER complexity_rank means simpler (fewer dependencies / knobs active).
    Returns the name of the simplest candidate within one SE of the best mean --
    Tier 2's pre-registered modified one-standard-error rule."""
    best_name = max(candidates, key=lambda n: candidates[n][0])
    best_mean, best_se, _ = candidates[best_name]
    within_one_se = [
        n for n, (mean, se, _rank) in candidates.items()
        if best_mean - mean <= max(best_se, se)
    ]
    return min(within_one_se, key=lambda n: candidates[n][2])


def _pooled_oof_auc_mean_se(rung_name):
    rows = RUNG_TABLES.get(rung_name)
    if rows is None or rows.empty or 'oof_auc' not in rows.columns:
        return None
    vals = rows['oof_auc'].to_numpy()
    if len(vals) < 2:
        return None
    return float(vals.mean()), float(vals.std(ddof=1) / np.sqrt(len(vals)))


# S1 vs S1b: S1 is simpler (no SSL node-LSTM checkpoint dependency) -> lower rank.
_s1 = _pooled_oof_auc_mean_se('S1_flip')
_s1b = _pooled_oof_auc_mean_se('S1b_ssl')
if _s1 is not None and _s1b is not None:
    candidates = {
        'S1_flip': (_s1[0], _s1[1], 0),
        'S1b_ssl': (_s1b[0], _s1b[1], 1),
    }
    selected = one_se_tie_breaker(candidates)
    print('Tier-2 one-SE tie-breaker over {S1_flip, S1b_ssl}:')
    for n, (mean, se, rank) in candidates.items():
        print(f"  {n}: pooled OOF AUC mean={mean:.4f} SE={se:.4f} complexity_rank={rank}")
    print(f"  -> selected: {selected} (primary arm; the other is retained as a "
          f"documented sensitivity arm, secondary at Tier 4)")
else:
    print('S1 and/or S1b OOF rows not available yet -- tie-breaker not computed.')


Tier-2 one-SE tie-breaker over {S1_flip, S1b_ssl}:
  S1_flip: pooled OOF AUC mean=0.7488 SE=0.0016 complexity_rank=0
  S1b_ssl: pooled OOF AUC mean=0.7502 SE=0.0062 complexity_rank=1
  -> selected: S1_flip (primary arm; the other is retained as a documented sensitivity arm, secondary at Tier 4)


## Tier 3 — robustness vetoes

Thresholds are fixed in `DOCS/temporal-first-ablation.md`'s addendum and never adjusted after seeing a result.

In [13]:
VETO_THRESHOLDS = {
    'threshold_sd': 0.15,      # SD of best_threshold across 5 folds x 4 seeds
    'ece_oof_cal': 0.10,       # temperature-scaled OOF ECE
    'scan_count_spearman': 0.3,  # |r| of prob vs n_scans, within-stable
}


def veto_row(name, run_dirs):
    df = rung_oof_rows(run_dirs)
    if df.empty:
        return {'rung': name, 'n_seeds': 0}
    thresholds = []
    for run_dir in run_dirs.values():
        summary = load_summary(run_dir)
        if summary is None:
            continue
        thresholds.extend(summary.get('cv_results', {}).get('best_threshold', []))
    threshold_sd = float(np.std(thresholds, ddof=1)) if len(thresholds) > 1 else float('nan')

    row = {
        'rung': name,
        'threshold_sd': threshold_sd,
        'threshold_sd_veto': threshold_sd > VETO_THRESHOLDS['threshold_sd'],
        'ece_oof_cal': df['ece_oof_cal'].mean() if 'ece_oof_cal' in df else float('nan'),
    }
    row['ece_veto'] = (row['ece_oof_cal'] > VETO_THRESHOLDS['ece_oof_cal']
                        if pd.notna(row['ece_oof_cal']) else None)
    if 'oof_prob_nscans_spearman_non_converter' in df.columns:
        r = df['oof_prob_nscans_spearman_non_converter'].mean()
        row['scan_count_spearman_non_converter'] = r
        row['scan_count_veto'] = abs(r) > VETO_THRESHOLDS['scan_count_spearman'] if pd.notna(r) else None
    if 'cohort_probe_auc' in df.columns:
        cpa = df['cohort_probe_auc'].mean()
        row['cohort_probe_auc'] = cpa
        row['cohort_probe_escalation'] = cpa > 0.75 if pd.notna(cpa) else None
    demo_auc_by_cohort = {c: DEMO_FLOOR[c].mean() for c in DEMO_FLOOR.columns
                           if c.startswith('oof_auc_') and c != 'oof_auc'} if not DEMO_FLOOR.empty else {}
    for c, demo_auc in demo_auc_by_cohort.items():
        if c in df.columns:
            row[f'{c}_vs_demo_floor'] = df[c].mean() - demo_auc
            row[f'{c}_collapse_veto'] = df[c].mean() < demo_auc
    return row


VETO_TABLE = pd.DataFrame([veto_row(name, dirs) for name, dirs in RUNG_RUN_DIRS.items()]).set_index('rung')
VETO_TABLE


,threshold_sd,threshold_sd_veto,ece_oof_cal,ece_veto,scan_count_spearman_non_converter,scan_count_veto,cohort_probe_auc,cohort_probe_escalation,oof_auc_adni_vs_demo_floor,oof_auc_adni_collapse_veto,oof_auc_delcode_vs_demo_floor,oof_auc_delcode_collapse_veto
rung,,,,,,,,,,,,
S0a_logreg_drift,0.165295,True,0.099211,False,-0.089585,False,NaN,None,0.040175,False,0.394222,False
S0_demo,0.034039,False,0.139733,True,-0.035833,False,NaN,None,0.000000,False,0.000000,False
S0b_gelstm_frozen,0.161187,True,0.122919,True,-0.312215,True,NaN,None,-0.019040,True,0.399115,False
S0c_gelstm_random,0.020113,False,0.130737,True,0.022954,False,NaN,None,0.018588,False,0.088013,False
S0d_braintokengt,0.158694,True,0.129853,True,-0.036283,False,NaN,None,0.103199,False,0.103099,False
S1_flip,0.123349,False,0.116005,True,-0.366478,True,0.859975,True,0.136424,False,0.355429,False
S1b_ssl,0.175876,True,0.103738,True,-0.314972,True,0.889796,True,0.146230,False,0.352050,False
S1c_recon_invalid,0.018351,False,0.135737,True,-0.068799,False,0.860954,True,0.040175,False,0.014911,False
S1c_recon_random,0.017727,False,0.130257,True,-0.117503,False,0.860028,True,0.017351,False,0.040308,False


## Scan-count-shortcut mechanism (kept arms)

`common.visit_confound.within_subject_prob_slopes` needs a reloaded model + the `per_visit_probs` hook, not just the OOF frame -- run on the CV pool (never the test set) for a specific kept arm's best-fold checkpoint by setting `MECHANISM_CHECK_EXP_ID` below.

In [14]:
MECHANISM_CHECK_EXP_ID = None  # e.g. 'tfgn-s1b-ssl-pooled-seed42' -- set to run this cell

if MECHANISM_CHECK_EXP_ID:
    from adapters import get_adapter
    from common.visit_confound import within_subject_prob_slopes
    from DATA.DELCODE.src.splitting.load_splits import splits_dir

    run_dir = resolve_source_run(MECHANISM_CHECK_EXP_ID, classifier_root=model_root)
    summary = load_summary(run_dir)
    if summary is None:
        raise FileNotFoundError(f'{MECHANISM_CHECK_EXP_ID}: no run_summary.json yet.')
    gaae_hp_path = model_root / 'configs' / 'gaae_delcode_whole_brain.json'
    gaae_hp = json.loads(gaae_hp_path.read_text()) if gaae_hp_path.is_file() else {}

    pooled_splits = repo_root / 'DATA' / 'POOLED_ADNI_DELCODE' / 'SPLITS' / 'downstream'
    cv_pool_df = pd.concat([pd.read_csv(pooled_splits / 'train.csv'),
                            pd.read_csv(pooled_splits / 'val.csv')], ignore_index=True)

    adapter_key = str(summary.get('model_config', {}).get('model_type', '')).lower()
    # model_type is the class name; map every family, not only the TFGN one --
    # the matched-window reference arm (PLAN.md section F) is a GELSTM, and an
    # unmapped key fails only at get_adapter(), after the frame is already built.
    adapter_key = {'tfgnclassifier': 'tfgn', 'logregdriftadapter': 'logregdrift',
                   'gelstmclassifier': 'gelstm',
                   'braintokengtclassifier': 'braintokengt'}.get(adapter_key, adapter_key)
    adapter = get_adapter(adapter_key)(
        gaae_ckpt_path=summary.get('gaae_checkpoint') or '', gaae_hp=gaae_hp,
        train_config=summary['training_config'],
        data_root=str(repo_root / 'DATA/POOLED_ADNI_DELCODE/__fc_wholebrain_sch200_flat__/matrices'),
        cohorts_csv=None, device='cpu', rng=None,
    )
    state = adapter.load_state(run_dir)
    cv_bundle = adapter.prepare_data(cv_pool_df)
    slope_df, slope_stats = within_subject_prob_slopes(cv_bundle, adapter.per_visit_probs, state, device='cpu')
    print(f'Within-subject slope of P(converter) vs visit index -- {MECHANISM_CHECK_EXP_ID} (CV pool, not test):')
    for grp, s in slope_stats.items():
        print(f"  {grp:14s} median_slope={s['median_slope']:.4f}  frac_negative={s['frac_negative']}  n={s['n']}")
else:
    print('MECHANISM_CHECK_EXP_ID not set -- skipping (set it to a kept arm\'s seed id to run).')


MECHANISM_CHECK_EXP_ID not set -- skipping (set it to a kept arm's seed id to run).


## Gate-map validation (S2, S5) -- pre-registered, wired once those rungs exist

Permutation null, cross-fold Spearman stability, per-cohort split (`DOCS/temporal-first-ablation.md` "Gate-map validation") -- no-op until `gate_scores.npy` exists (S2 `use_gate: true` onward).

In [15]:
GATE_RUNG = 'S2_gate'  # update once S2 is in RUNG_PREFIXES

if GATE_RUNG in RUNG_RUN_DIRS and RUNG_RUN_DIRS[GATE_RUNG]:
    gate_maps = []
    for exp_id, run_dir in RUNG_RUN_DIRS[GATE_RUNG].items():
        gp = run_dir / 'gate_scores.npy'
        if gp.is_file():
            gate_maps.append((exp_id, np.load(gp)))
    print(f'{len(gate_maps)} gate maps found for {GATE_RUNG}.')
    # Permutation null / cross-fold Spearman / per-cohort split go here once S2 exists
    # -- structure only, not computed on data that doesn't exist yet.
else:
    print(f'{GATE_RUNG}: no runs yet -- gate-map validation is a no-op until S2/S5 land.')


4 gate maps found for S2_gate.


## Tier 4 — frozen reads (in-domain test + OASIS-3, exactly once)

**Gated.** Nothing below executes unless `RUN_FROZEN_READ = True` and `FROZEN_WINNER_ID` names the frozen winning arm's id prefix (no seed suffix -- all 4 seeds are read). Uses `common.frozen_read.score_frozen_split` -- reloads each seed's saved checkpoint, scores at its own OOF-derived threshold, records via the same `record_test_metrics` / `record_external_metrics` every non-deferred run already uses.

In [16]:
def run_frozen_reads(id_prefix, label):
    """One Tier-4 frozen-read pass (in-domain test + OASIS-3, all 4 seeds) for a
    single arm. `label` is 'PRIMARY' or 'SECONDARY (sensitivity arm, not primary)' --
    printed on every line so a reader of run.log / notebook output can never mistake
    a secondary sensitivity read for the primary estimate."""
    from common.frozen_read import score_frozen_split
    from DATA.DELCODE.src.splitting.load_splits import splits_dir  # noqa: F401

    gaae_hp_path = model_root / 'configs' / 'gaae_delcode_whole_brain.json'
    gaae_hp = json.loads(gaae_hp_path.read_text()) if gaae_hp_path.is_file() else {}
    pooled_dir = repo_root / 'DATA' / 'POOLED_ADNI_DELCODE'
    in_domain_test_df = pd.read_csv(pooled_dir / 'SPLITS' / 'downstream' / 'test.csv')
    oasis_splits = repo_root / 'DATA' / 'OASIS3' / '__metadata__' / 'SPLITS' / 'downstream'
    oasis_test_df = pd.concat(
        [pd.read_csv(oasis_splits / f'{s}.csv') for s in ('train', 'val', 'test')], ignore_index=True,
    )
    oasis_test_df['cohort'] = 'oasis3'

    results = {}
    for seed in SEEDS:
        exp_id = f'{id_prefix}-seed{seed}'
        run_dir = resolve_source_run(exp_id, classifier_root=model_root)
        summary = load_summary(run_dir)
        if summary is None:
            raise FileNotFoundError(f'{exp_id}: no run_summary.json yet -- not ready for a frozen read.')
        adapter_key = str(summary.get('model_config', {}).get('model_type', '')).lower()
        # model_type is the class name; map every family, not only the TFGN one --
        # the matched-window reference arm (PLAN.md section F) is a GELSTM, and an
        # unmapped key fails only at get_adapter(), after the frame is already built.
        adapter_key = {'tfgnclassifier': 'tfgn', 'logregdriftadapter': 'logregdrift',
                       'gelstmclassifier': 'gelstm',
                       'braintokengtclassifier': 'braintokengt'}.get(adapter_key, adapter_key)
        common_kwargs = dict(
            adapter_key=adapter_key,
            data_root=str(pooled_dir / '__fc_wholebrain_sch200_flat__' / 'matrices'),
            cohorts_csv=None, gaae_ckpt_path=summary.get('gaae_checkpoint') or '',
            gaae_hp=gaae_hp, device='cpu',
        )
        test_metrics = score_frozen_split(run_dir, in_domain_test_df, record_as='test', **common_kwargs)
        ext_metrics = score_frozen_split(run_dir, oasis_test_df, record_as='external', cohort='oasis3', **common_kwargs)
        results[exp_id] = {'test_auc': test_metrics['auc'], 'ext_oasis3_auc': ext_metrics['auc']}
        print(f'[{label}] {exp_id}: test_auc={test_metrics["auc"]:.4f}  ext_oasis3_auc={ext_metrics["auc"]:.4f}')

    frozen_df = pd.DataFrame(results).T
    print()
    print(f'[{label}] Frozen reads across seeds ({id_prefix}):')
    print(frozen_df)
    print()
    print(f"[{label}] In-domain test AUC: {frozen_df['test_auc'].mean():.4f} +/- {frozen_df['test_auc'].std():.4f}")
    print(f"[{label}] OASIS-3 AUC:        {frozen_df['ext_oasis3_auc'].mean():.4f} +/- {frozen_df['ext_oasis3_auc'].std():.4f}")

    winner_oof = RUNG_TABLES.get(id_prefix)
    if winner_oof is None:
        for name, prefix in RUNG_PREFIXES.items():
            if prefix == id_prefix:
                winner_oof = RUNG_TABLES.get(name)
    if winner_oof is not None and not winner_oof.empty:
        se_oof = winner_oof['oof_auc'].std(ddof=1) / np.sqrt(len(winner_oof))
        se_test = frozen_df['test_auc'].std(ddof=1) / np.sqrt(len(frozen_df)) if len(frozen_df) > 1 else float('nan')
        half_width = 1.96 * np.sqrt(se_oof ** 2 + se_test ** 2)
        lo, hi = winner_oof['oof_auc'].mean() - half_width, winner_oof['oof_auc'].mean() + half_width
        consistent = lo <= frozen_df['test_auc'].mean() <= hi
        print()
        print(f'[{label}] Transport check: OOF={winner_oof["oof_auc"].mean():.4f}  '
              f'95% prediction interval=[{lo:.4f}, {hi:.4f}]  '
              f'test={frozen_df["test_auc"].mean():.4f}  '
              f'-> {"consistent" if consistent else "inconsistent"} with CV->test transport.')
        print(f'[{label}] Winner\'s-curse statement: the OOF AUC above is expected to be optimistic '
              '(selected as the best of the ladder); the frozen test read above is the '
              'unbiased estimate. Report both, not the OOF number alone, as the headline.')
    return frozen_df


if not RUN_FROZEN_READ:
    print('RUN_FROZEN_READ=False -- Tier 4 skipped (the ladder is not frozen yet, or '
          'this is a routine re-run of sections 1-6). Flip both flags above once ready.')
elif not FROZEN_WINNER_ID:
    raise ValueError('RUN_FROZEN_READ=True requires FROZEN_WINNER_ID (an id prefix).')
else:
    FROZEN_RESULTS = run_frozen_reads(FROZEN_WINNER_ID, 'PRIMARY')
    # SECONDARY_SENSITIVITY_ID: a single id prefix or a list of them, each read in
    # this same one-shot pass and reported side by side -- never substituted for the
    # primary. S5's label is specialised so its number reads as the interpretability
    # layer's estimate (PLAN.md section A: kept regardless of AUC), not a rival to S1.
    _SECONDARY_LABELS = {
        'tfgn-s5-dualscore-pooled': 'SECONDARY (interpretability layer, kept regardless of AUC -- not a competing endpoint)',
    }
    SECONDARY_RESULTS = {}
    _secondary_ids = SECONDARY_SENSITIVITY_ID if SECONDARY_SENSITIVITY_ID else []
    if isinstance(_secondary_ids, str):
        _secondary_ids = [_secondary_ids]
    for _sec_id in _secondary_ids:
        print()
        print('=' * 70)
        _label = _SECONDARY_LABELS.get(_sec_id, 'SECONDARY (sensitivity arm, not primary)')
        SECONDARY_RESULTS[_sec_id] = run_frozen_reads(_sec_id, _label)


LongitudinalSubjectDataset[v2][adni]: 39 subjects (13 converter, 26 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=8  mean=3.2


LongitudinalSubjectDataset[v2][delcode]: 25 subjects (11 converter, 14 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=5  mean=3.0


LongitudinalSubjectDataset[v2][oasis3]: 60 subjects (31 converter, 29 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=6  mean=2.6


[PRIMARY] tfgn-s1-flip-pooled-seed42: test_auc=0.7740  ext_oasis3_auc=0.4705
LongitudinalSubjectDataset[v2][adni]: 39 subjects (13 converter, 26 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=8  mean=3.2


LongitudinalSubjectDataset[v2][delcode]: 25 subjects (11 converter, 14 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=5  mean=3.0


...

LongitudinalSubjectDataset[v2][oasis3]: 60 subjects (31 converter, 29 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=6  mean=2.6


[SECONDARY (interpretability layer, kept regardless of AUC -- not a competing endpoint)] tfgn-s5-dualscore-pooled-seed44: test_auc=0.7688  ext_oasis3_auc=0.5217
LongitudinalSubjectDataset[v2][adni]: 39 subjects (13 converter, 26 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=8  mean=3.2


LongitudinalSubjectDataset[v2][delcode]: 25 subjects (11 converter, 14 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=5  mean=3.0


LongitudinalSubjectDataset[v2][oasis3]: 60 subjects (31 converter, 29 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=6  mean=2.6


[SECONDARY (interpretability layer, kept regardless of AUC -- not a competing endpoint)] tfgn-s5-dualscore-pooled-seed45: test_auc=0.7937  ext_oasis3_auc=0.5083

[SECONDARY (interpretability layer, kept regardless of AUC -- not a competing endpoint)] Frozen reads across seeds (tfgn-s5-dualscore-pooled):
                                 test_auc  ext_oasis3_auc
tfgn-s5-dualscore-pooled-seed42  0.801042        0.497219
tfgn-s5-dualscore-pooled-seed43  0.784375        0.500556
tfgn-s5-dualscore-pooled-seed44  0.768750        0.521691
tfgn-s5-dualscore-pooled-seed45  0.793750        0.508343

[SECONDARY (interpretability layer, kept regardless of AUC -- not a competing endpoint)] In-domain test AUC: 0.7870 +/- 0.0139
[SECONDARY (interpretability layer, kept regardless of AUC -- not a competing endpoint)] OASIS-3 AUC:        0.5070 +/- 0.0109

[SECONDARY (interpretability layer, kept regardless of AUC -- not a competing endpoint)] Transport check: OOF=0.7331  95% prediction interval=[0.7113

## Guard check — sections 1-6 touched no test/external metric

In [17]:
_forbidden = {'TEST_METRICS', 'EXTERNAL_METRICS', 'test_df', 'in_domain_test_df', 'oasis_test_df'}
_touched = _forbidden & set(dir())
if RUN_FROZEN_READ:
    print(f'RUN_FROZEN_READ=True -- Tier 4 ran by design; test/external names present: {_touched or "(pandas frames only, as expected)"}.')
else:
    _unexpected = _touched - {'test_df'}  # 'test_df' would only exist if RUN_FROZEN_READ ran
    assert not _unexpected, f'Sections 1-6 touched test/external state unexpectedly: {_unexpected}'
    print('OK -- no test/external metric was read (RUN_FROZEN_READ=False).')


RUN_FROZEN_READ=True -- Tier 4 ran by design; test/external names present: (pandas frames only, as expected).
